In [1]:
import os
os.getcwd()

'/Users/mariappan.subramanian/Documents/repo/forked/meridian/demo'

In [2]:
import arviz as az
import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import formatter
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.data import data_frame_input_data_builder
from meridian.data import test_utils
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
import numpy as np
import pandas as pd
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp

ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print(
    'Num GPUs Available: ',
    len(tf.config.experimental.list_physical_devices('GPU')),
)
print(
    'Num CPUs Available: ',
    len(tf.config.experimental.list_physical_devices('CPU')),
)

Your runtime has 25.8 gigabytes of available RAM

Num GPUs Available:  0
Num CPUs Available:  1


1. National Model - Both Media & RF

In [3]:
df = pd.read_csv("./../meridian/data/simulated_data/csv/national_media_rf.csv")
df.head()

,time,conversions,revenue_per_conversion,Channel0_impression,Channel1_impression,Channel2_impression,Channel3_impression,Channel0_spend,Channel1_spend,Channel2_spend,Channel3_spend,Channel3_reach,Channel3_frequency,competitor_activity_score_control,sentiment_score_control
0,2021-01-25,224655360,0.034894,29203792,8887214,6888124,913520,324688.25,101612.82,82259.586,10743.550,728787,1.253480,-0.416371,0.683978
1,2021-02-01,158389250,0.035067,24655420,13865920,13265415,6044235,274119.34,158537.33,158418.690,71083.875,4242109,1.424818,-0.846952,-1.381805
2,2021-02-08,216225630,0.034893,31907648,19497852,16535126,11904464,354749.75,222930.58,197466.340,140003.730,6826023,1.743982,-1.098846,0.601687
3,2021-02-15,203125630,0.035203,28151956,9928580,5394670,4685435,312993.90,113519.38,64424.410,55103.562,3200847,1.463811,-0.779951,-0.330715
4,2021-02-22,222105420,0.035005,23894308,14377868,14984190,14355571,265657.30,164390.75,178944.700,168830.230,7487385,1.917301,-0.982015,0.638006


In [4]:
model_config = {

  # time and geo inputs
  'time_col': 'week',
  # 'geo_col': 'geo',  # assumed to be national model if not given
  # 'population_col': 'population', # mandatory if geo_col is given

  # kpi inputs
  'kpi_col': 'conversions',  #
  'kpi_type': 'non_revenue',
  'revenue_per_kpi_col': 'revenue_per_conversion',  # needed if kpi_type is non_revenue

  # impression based media inputs
  'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
  'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
  'media_channels': ['Channel0', 'Channel1', 'Channel2'],

  # reach based media inputs
  'reach_cols': ['Channel3_reach'],
  'frequency_cols': ['Channel3_frequency'],
  'rf_spend_cols': ['Channel3_spend'],
  'rf_channels': ['Channel3'],

  # control inputs
  'control_cols': ["sentiment_score_control", "competitor_activity_score_control"]

  }

In [5]:
# Create a DataFrameInputDataBuilder instance.
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type='non_revenue',
    default_kpi_column='conversions',
    default_revenue_per_kpi_column='revenue_per_conversion',
)

# Offer the components to the builder. Note that the components may be offered all at once or piecewise.
builder = (
    builder.with_kpi(df, geo_col=model_config.get('geo_col'))
    .with_revenue_per_kpi(df, geo_col=model_config.get('geo_col'))
)

if model_config.get('population_col'):
  builder = builder.with_population(
    df, population_col=model_config['population_col'], geo_col=model_config.get('geo_col')
  )

if model_config.get('control_cols'):
  builder = builder.with_controls(
      df, control_cols=model_config['control_cols'], geo_col=model_config.get('geo_col')
  )

if model_config.get('media_cols'):
  builder = builder.with_media(
      df,
      media_cols=model_config['media_cols'],
      media_spend_cols=model_config['media_spend_cols'],
      media_channels=model_config['media_channels'],
      geo_col=model_config.get('geo_col')
  )

if model_config.get('reach_cols'):
  builder = builder.with_reach(
      df,
      reach_cols=model_config['reach_cols'],
      frequency_cols=model_config['frequency_cols'],
      rf_spend_cols=model_config['rf_spend_cols'],
      rf_channels=model_config['rf_channels'],
      geo_col=model_config.get('geo_col')
  )

# finally do the data build
input_data = builder.build()

In [6]:
input_data.media

<xarray.DataArray 'media' (geo: 1, media_time: 156, media_channel: 3)> Size: 4kB
array([[[29203792,  8887214,  6888124],
        [24655420, 13865920, 13265415],
        [31907648, 19497852, 16535126],
        [28151956,  9928580,  5394670],
        [23894308, 14377868, 14984190],
        [14481371, 15537686, 17689056],
        [24939800, 13380116, 19709212],
        [ 8301820,  7176677, 16582619],
        [28776724,  8694681, 22667478],
        [24183782, 13468196, 13229485],
        [29492328,  9511647,  5209152],
        [26652520, 18312648, 16359222],
        [26461152, 16221465, 13559208],
        [24423240, 16356870, 12072692],
        [20516068, 20833432, 12924195],
        [10099865, 13555704,  3858469],
        [21859196, 17441450, 13411954],
        [35513180, 24046754, 14952220],
        [20534680, 13432045,  9071299],
        [23742996, 14348278, 11613460],
...
        [18781202, 18118548, 14910117],
        [24290264, 16079873, 25542668],
        [28033248, 12034335, 11441497],
        [23764412, 17199160, 18666006],
        [29605884, 18641624,  9499634],
        [23326030, 15609850, 12566632],
        [31240002, 20084206,  5409317],
        [28164296,  6798141, 18377970],
        [33700010, 15593805, 15497353],
        [15927216, 16060470,  4538802],
        [24444068, 12646286, 17420886],
        [21991204, 11876869, 12317416],
        [19703536,  9102192, 14174364],
        [21892804, 12925465, 10848844],
        [22173538, 12139883,    82574],
        [35425350, 17893086,  9641156],
        [24527896, 10201574, 15765894],
        [22461000, 19979368,  2983733],
        [14518918,  5393032, 14088305],
        [24398364, 29270078, 13330652]]])
Coordinates:
  * media_time     (media_time) object 1kB '2021-01-25' ... '2024-01-15'
  * media_channel  (media_channel) object 24B 'Channel0' 'Channel1' 'Channel2'
  * geo            (geo) <U12 48B 'national_geo'

In [19]:
# model configuration
model_spec = spec.ModelSpec()
mmm = model.Meridian(input_data=input_data, model_spec=model_spec)

/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/model.py:67: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(
I0000 00:00:1757516504.964398   24772 service.cc:148] XLA service 0x110e5dfd0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1757516504.964468   24772 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1757516504.979138   24772 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [20]:
%%time
mmm.sample_prior(500)
mmm.sample_posterior(
    n_chains=10, n_adapt=2000, n_burnin=500, n_keep=1000, seed=0
)

/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/prior_distribution.py:1146: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. tau_g_excl_baseline has been automatically set to Deterministic(0).
  warnings.warn(
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/prior_distribution.py:1146: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_m has been automatically set to Deterministic(0).
  warnings.warn(
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/prior_distribution.py:1146: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_rf has been automatically set to Deterministic(0).
  warnings.warn(
/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/model/prior_distribution.py:1146: UserWarning: Hierarchical distribution parameters must

CPU times: user 1min 58s, sys: 3.65 s, total: 2min 2s
Wall time: 2min 2s


In [21]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

/Users/mariappan.subramanian/Documents/repo/forked/meridian/meridian/analysis/analyzer.py:590: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


alt.LayerChart(...)

In [22]:
# save the model
file_path = './saved_models/demo_model_national_all_channels.pkl'
model.save_mmm(mmm, file_path)

In [23]:
mmm.media_tensors.media.shape

TensorShape([1, 156, 3])

In [24]:
mmm.population

<tf.Tensor: shape=(1,), dtype=float32, numpy=array([1.], dtype=float32)>

2. Geo Model With Both Media & RF

In [3]:
df = pd.read_csv("./../meridian/data/simulated_data/csv/geo_media_rf.csv")
df.head()

,geo,time,Channel0_impression,Channel1_impression,Channel2_impression,Channel3_impression,competitor_activity_score_control,sentiment_score_control,Channel0_spend,Channel1_spend,Channel2_spend,Channel3_spend,conversions,revenue_per_conversion,population,Channel3_reach,Channel3_frequency
0,Geo0,2021-01-25,1392518.0,3733.0,670235.0,0.0,-0.783350,3.036792,15482.038,42.681618,8004.1030,0.0000,12530976.0,0.035213,487878.0,0.0,0.000000
1,Geo0,2021-02-01,937228.0,722210.0,745025.0,226872.0,-1.834407,-4.244738,10420.116,8257.458000,8897.2630,2668.1526,4926880.5,0.035026,487878.0,181472.0,1.250176
2,Geo0,2021-02-08,1286569.0,329778.0,786262.0,743321.0,-1.995511,0.200945,14304.095,3770.548600,9389.7250,8741.9060,10300557.0,0.034529,487878.0,381866.0,1.946549
3,Geo0,2021-02-15,1149907.0,529628.0,190449.0,400702.0,-4.925971,-1.542391,12784.685,6055.552700,2274.3865,4712.4990,7398028.5,0.035430,487878.0,273012.0,1.467708
4,Geo0,2021-02-22,1077028.0,1057061.0,584113.0,346168.0,-4.541369,0.668494,11974.415,12086.009000,6975.6143,4071.1460,9409540.0,0.034637,487878.0,247506.0,1.398625


In [4]:
model_config = {

  # time and geo inputs
  'time_col': 'week',
  'geo_col': 'geo',  # assumed to be national model if not given
  'population_col': 'population', # mandatory if geo_col is given

  # kpi inputs
  'kpi_col': 'conversions',  #
  'kpi_type': 'non_revenue',
  'revenue_per_kpi_col': 'revenue_per_conversion',  # needed if kpi_type is non_revenue

  # impression based media inputs
  'media_cols': ['Channel0_impression', 'Channel1_impression', 'Channel2_impression'],
  'media_spend_cols': ['Channel0_spend', 'Channel1_spend', 'Channel2_spend'],
  'media_channels': ['Channel0', 'Channel1', 'Channel2'],

  # reach based media inputs
  'reach_cols': ['Channel3_reach'],
  'frequency_cols': ['Channel3_frequency'],
  'rf_spend_cols': ['Channel3_spend'],
  'rf_channels': ['Channel3'],

  # control inputs
  'control_cols': ["sentiment_score_control", "competitor_activity_score_control"]

  }

In [5]:
# Create a DataFrameInputDataBuilder instance.
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type='non_revenue',
    default_kpi_column='conversions',
    default_revenue_per_kpi_column='revenue_per_conversion',
)

# Offer the components to the builder. Note that the components may be offered all at once or piecewise.
builder = (
    builder.with_kpi(df, geo_col=model_config.get('geo_col'))
    .with_revenue_per_kpi(df, geo_col=model_config.get('geo_col'))
)

if model_config.get('population_col'):
  builder = builder.with_population(
    df, population_col=model_config['population_col'], geo_col=model_config.get('geo_col')
  )

if model_config.get('control_cols'):
  builder = builder.with_controls(
      df, control_cols=model_config['control_cols'], geo_col=model_config.get('geo_col')
  )

if model_config.get('media_cols'):
  builder = builder.with_media(
      df,
      media_cols=model_config['media_cols'],
      media_spend_cols=model_config['media_spend_cols'],
      media_channels=model_config['media_channels'],
      geo_col=model_config.get('geo_col')
  )

if model_config.get('reach_cols'):
  builder = builder.with_reach(
      df,
      reach_cols=model_config['reach_cols'],
      frequency_cols=model_config['frequency_cols'],
      rf_spend_cols=model_config['rf_spend_cols'],
      rf_channels=model_config['rf_channels'],
      geo_col=model_config.get('geo_col')
  )

# finally do the data build
input_data = builder.build()

In [39]:
# model configuration
model_spec = spec.ModelSpec()
mmm = model.Meridian(input_data=input_data, model_spec=model_spec)

In [26]:
mmm.knot_info.knot_locations

array([  0,  17,  34,  51,  68,  86, 103, 120, 137, 155])

In [37]:
knot_weights = mmm.knot_info.weights.T  # (n_times, n_knots)
knot_weights[-3, ]

array([0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.11111111, 0.8888889 ],
      dtype=float32)

In [44]:
%%time
mmm.sample_prior(500)
mmm.sample_posterior(
    n_chains=10, n_adapt=2000, n_burnin=500, n_keep=1000, seed=0
)

W0000 00:00:1757517485.575375   24772 assert_op.cc:38] Ignoring Assert operator mcmc_retry_init/assert_equal_1/Assert/AssertGuard/Assert


CPU times: user 1h 7min 9s, sys: 24min 42s, total: 1h 31min 51s
Wall time: 47min 50s


In [45]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

alt.LayerChart(...)

In [46]:
# save the model
file_path = './saved_models/demo_model_geo_all_channels_new.pkl'
model.save_mmm(mmm, file_path)

In [124]:
# --------------------------------------- Budget Optimization and Save the Results---------------------------------------------------------------------- #
model_name = 'demo_model_national_all_channels.pkl'
model_path = f'./saved_models/{model_name}'
mmm = model.load_mmm(model_path)
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()
model.save_mmm(optimization_results, f'./saved_opts/{model_name}')

/Users/mariappan.subramanian/Documents/repo/forked/meridian/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/Users/mariappan.subramanian/Documents/repo/forked/meridian/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


In [47]:
# ---------------------------------------------------------------------------------------------------------------------- #